In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "ETHUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "xgb"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from functools import partial
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split, information_coefficient, rank_information_coefficient
from models import OBJECTIVES, MODEL_REGISTRY

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_1m.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")
features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 284,679


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,dow_cos,dom_sin,dom_cos,month_sin,month_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-09-01 00:00:00+00:00,4391.83,4391.83,4386.25,4389.95,325.8367,2025-09-01 00:00:59.999999+00:00,1.429921e+06,3320,111.1554,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.000000,0.000000,0.000000,NaN,NaN
1,2025-09-01 00:01:00+00:00,4389.96,4391.40,4389.68,4391.16,158.5513,2025-09-01 00:01:59.999999+00:00,6.961133e+05,1908,95.8326,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.027147,0.015082,0.012066,NaN,NaN
2,2025-09-01 00:02:00+00:00,4391.16,4391.16,4386.14,4388.19,187.0756,2025-09-01 00:02:59.999999+00:00,8.207380e+05,3039,100.6024,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.057508,-0.014668,-0.042840,NaN,NaN
3,2025-09-01 00:03:00+00:00,4388.19,4389.97,4386.33,4386.45,341.8429,2025-09-01 00:03:59.999999+00:00,1.500188e+06,2817,177.2970,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.157424,-0.063027,-0.094397,NaN,NaN
4,2025-09-01 00:04:00+00:00,4386.45,4386.45,4375.39,4376.57,622.3295,2025-09-01 00:04:59.999999+00:00,2.725730e+06,5777,183.8020,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.601551,-0.223226,-0.378325,NaN,NaN


In [8]:
target_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 284,601
[info] optuna train rows: 182,144
[info] valid rows:        45,536
[info] test rows:         56,921


In [9]:
study = optuna.create_study(direction="maximize")
objective_fn = partial(
    OBJECTIVES[MODEL_TYPE],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

study.optimize(objective_fn, n_trials=50, show_progress_bar=True)

print("\n[optuna] best trial")
print(f"value: {study.best_value:.6f}")
print("params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-19 21:34:01,486] A new study created in memory with name: no-name-dff78351-003d-44b6-bba5-5dbc58cfee67


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:06<?, ?it/s]

Best trial: 0. Best value: 0.0127017:   0%|          | 0/50 [00:06<?, ?it/s]

Best trial: 0. Best value: 0.0127017:   2%|▏         | 1/50 [00:06<05:10,  6.33s/it]

...

Best trial: 4. Best value: 0.0283372:  98%|█████████▊| 49/50 [03:26<00:04,  4.81s/it]

Best trial: 4. Best value: 0.0283372:  98%|█████████▊| 49/50 [03:26<00:04,  4.81s/it]

Best trial: 4. Best value: 0.0283372: 100%|██████████| 50/50 [03:26<00:00,  4.94s/it]

Best trial: 4. Best value: 0.0283372: 100%|██████████| 50/50 [03:26<00:00,  4.13s/it]

[I 2026-03-19 21:37:28,115] Trial 49 finished with value: 0.012816250573216417 and parameters: {'n_estimators': 1200, 'max_depth': 7, 'learning_rate': 0.02020794606117489, 'subsample': 0.8938708648629872, 'colsample_bytree': 0.8764624109141421, 'min_child_weight': 4, 'reg_alpha': 4.1601758696540186e-08, 'reg_lambda': 0.08058912578733275}. Best is trial 4 with value: 0.028337228213957392.

[optuna] best trial
value: 0.028337
params:
  n_estimators: 200
  max_depth: 5
  learning_rate: 0.004294772229492131
  subsample: 0.967904901741313
  colsample_bytree: 0.6996891175283455
  min_child_weight: 6
  reg_alpha: 0.0013895476641674074
  reg_lambda: 4.0293201445404344e-08


In [10]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1

X_train_full = train_df[feature_cols]
y_train_full = train_df[target_col]

final_model = MODEL_REGISTRY[MODEL_TYPE](**best_params)

start = time.time()
print(f"[training] fitting final {MODEL_TYPE}...")
final_model.fit(X_train_full, y_train_full)
print(f"[training] done in {time.time() - start:.2f}s")

[training] fitting final xgb...


[training] done in 40.50s


In [11]:
train_pred = final_model.predict(X_train_full)
test_pred = final_model.predict(X_test)

In [12]:
# evaluate
print("[eval] computing metrics...")
train_ic = information_coefficient(y_train_full.values, train_pred)
test_ic = information_coefficient(y_test.values, test_pred)

train_rank_ic = rank_information_coefficient(y_train_full.values, train_pred)
test_rank_ic = rank_information_coefficient(y_test.values, test_pred)

train_rmse = root_mean_squared_error(y_train_full, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print("\n===== RESULTS =====")
print(f"Train IC:      {train_ic:.6f}")
print(f"Test IC:       {test_ic:.6f}")
print(f"Train Rank IC: {train_rank_ic:.6f}")
print(f"Test Rank IC:  {test_rank_ic:.6f}")
print(f"Train RMSE:    {train_rmse:.6f}")
print(f"Test RMSE:     {test_rmse:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train IC:      0.275390
Test IC:       -0.013787
Train Rank IC: 0.069378
Test Rank IC:  -0.001375
Train RMSE:    0.002101
Test RMSE:     0.002300


In [13]:
# feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(importances)


===== FEATURE IMPORTANCE =====
volume_mom_5        0.043070
imbalance_5         0.042804
range_ratio         0.041530
vol_regime_ratio    0.040463
dom_sin             0.037926
trend_x_imb         0.037100
dow_cos             0.036447
mr_x_vol            0.033295
trades_z            0.029149
num_trades_mom_5    0.028052
hour_cos            0.028021
mom_5               0.027608
imbalance_15        0.026816
mom_10              0.026671
vol_ratio_5_30      0.026415
volume_z            0.026393
mom_15              0.024741
mom_60              0.024474
trend_strength      0.024355
mom_x_imb           0.023233
dist_ma_5           0.022724
dist_ma_15_z        0.020994
macd_hist           0.020494
dow_sin             0.020412
vol_5               0.020394
is_trending         0.020172
is_high_vol         0.019802
vol_15              0.019438
dom_cos             0.019232
bar_range           0.018752
vol_30              0.018441
mom_3               0.018285
imbalance           0.017349
hour_sin   

In [14]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/xgb/ETHUSDT__5_predictions.csv


In [15]:
# save model
joblib.dump(final_model, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# save feature importance
importances.to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(study.best_value),
    "model_params": best_params,
    "n_features": int(len(feature_cols)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": train_ic,
    "test_ic": test_ic,
    "train_rank_ic": train_rank_ic,
    "test_rank_ic": test_rank_ic,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat()
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/xgb/ETHUSDT__h5_model.joblib
[saved] features -> models/xgb/ETHUSDT__h5_feature_cols.json
[saved] feature importance -> models/xgb/ETHUSDT__h5_feature_importance.csv
[saved] metadata -> models/xgb/ETHUSDT__h5_meta.json
